## 2. MLP 

In [1]:
# imports
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import optuna
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
from scipy.ndimage import gaussian_filter1d
from itertools import permutations
from tqdm.auto import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [2]:
# reproducability

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


In [3]:
# load data

df_p1 = pd.read_json("data/raw_static/ARMAS_STATIC_20132023_P1.json", lines=True)
df_p2 = pd.read_json("data/raw_static/ARMAS_STATIC_20132023_P2.json", lines=True)
df_p3 = pd.read_json("data/raw_static/ARMAS_STATIC_20132023_P3.json", lines=True)

df_p1["partition"], df_p2["partition"], df_p3["partition"] = "P1", "P2", "P3"

df = pd.concat([df_p1, df_p2, df_p3], ignore_index=True)

print(df.shape); print(df["partition"].value_counts())

(92476, 49)
partition
P1    32380
P2    30787
P3    29309
Name: count, dtype: int64


In [4]:
# target 

target_col = "ARMAS"
drop_cols = ["ARMAS", "NAIRASV2", "NAIRASV3", "Datetime", "Vehicle_ID", "partition"]
feature_cols = [c for c in df.columns if c not in drop_cols]

splits = list(permutations(["P1", "P2", "P3"], 3))   # 6 rotations

print(len(feature_cols), "features,", len(splits), "rotations")

43 features, 6 rotations


In [5]:
# evaluation

def evaluate_regression(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    if len(y_true) > 1 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        pearson, r2 = pearsonr(y_true, y_pred)[0], r2_score(y_true, y_pred)
    else:
        pearson = r2 = np.nan
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2, "Pearson_r": pearson,
            "Bias": np.mean(y_pred - y_true),
            "Underestimation_%": np.mean(y_pred < y_true) * 100}

def regression_metrics_subset(df_subset):
    y_true, y_pred = df_subset["y_true"].values, df_subset["y_pred"].values
    out = {"n": len(df_subset),
           "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
           "MAE": mean_absolute_error(y_true, y_pred),
           "Bias": np.mean(y_pred - y_true),
           "Underestimation_%": np.mean(y_pred < y_true) * 100,
           "Median_AE": np.median(np.abs(y_pred - y_true))}
    if len(df_subset) > 1 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        out["R2"], out["Pearson_r"] = r2_score(y_true, y_pred), pearsonr(y_true, y_pred)[0]
    else:
        out["R2"] = out["Pearson_r"] = np.nan
    return pd.Series(out)

def compute_density_weights(y, n_bins=50, sigma=2.0, max_ratio=50.0):
    y = np.asarray(y, dtype=float)
    edges = np.linspace(y.min(), y.max() + 1e-9, n_bins + 1)
    idx = np.clip(np.digitize(y, edges) - 1, 0, n_bins - 1)
    counts = np.bincount(idx, minlength=n_bins).astype(float)
    smoothed = np.clip(gaussian_filter1d(counts, sigma=sigma, mode="nearest"), 1e-6, None)
    w = 1.0 / smoothed[idx]
    w = np.clip(w / w.min(), 1.0, max_ratio)
    return w * (len(w) / w.sum())

In [6]:
# mlp model and training

def make_mlp(n_features, n_layers, d_hidden, dropout):
    layers, d_in = [], n_features
    for _ in range(n_layers):
        layers += [nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Dropout(dropout)]
        d_in = d_hidden
    layers += [nn.Linear(d_in, 1)]
    return nn.Sequential(*layers)

def train_nn(model, X_tr, y_tr, X_va, y_va, w_tr=None,
             lr=1e-3, weight_decay=1e-5, batch_size=256,
             max_epochs=200, patience=16, device=DEVICE):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    Xtr = torch.tensor(X_tr, dtype=torch.float32)
    ytr = torch.tensor(y_tr, dtype=torch.float32).view(-1, 1)
    cols = [Xtr, ytr]
    if w_tr is not None:
        cols.append(torch.tensor(w_tr, dtype=torch.float32).view(-1, 1))
    dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(*cols),
                                     batch_size=batch_size, shuffle=True)
    Xva = torch.tensor(X_va, dtype=torch.float32).to(device)
    yva = torch.tensor(y_va, dtype=torch.float32).view(-1, 1).to(device)

    best_val, best_state, bad = np.inf, None, 0
    for _ in range(max_epochs):
        model.train()
        for batch in dl:
            opt.zero_grad()
            parts = [t.to(device) for t in batch]
            xb, yb = parts[0], parts[1]
            pred = model(xb)
            if w_tr is None:
                loss = ((pred - yb) ** 2).mean()
            else:
                loss = (parts[2] * (pred - yb) ** 2).mean()   # gewichteter MSE
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vloss = ((model(Xva) - yva) ** 2).mean().item()   # Early Stopping: plain val-MSE
        if vloss < best_val - 1e-6:
            best_val, bad = vloss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val

def predict_nn(model, X, device=DEVICE):
    model.eval()
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy().ravel()

In [7]:
# constants and optuna
N_TRIALS    = 40       # tuning budget per split
MAX_EPOCHS  = 200      # cap
PATIENCE    = 16

def make_objective(Xtr_s, ytr, Xva_s, yva, y_mean, y_std):
    #optuna goal: minimize validation rmse in µSv/h
    
    def objective(trial):
        n_layers   = trial.suggest_int("n_layers", 1, 5)
        d_hidden   = trial.suggest_categorical("d_hidden", [64, 128, 256, 512])
        dropout    = trial.suggest_float("dropout", 0.0, 0.5)
        lr         = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
        wd         = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

        seed_everything(42)
        model = make_mlp(Xtr_s.shape[1], n_layers, d_hidden, dropout)
        model, _ = train_nn(model, Xtr_s, ytr, Xva_s, yva,
                            lr=lr, weight_decay=wd, batch_size=batch_size,
                            max_epochs=MAX_EPOCHS, patience=PATIENCE)
        yva_pred = predict_nn(model, Xva_s) * y_std + y_mean       # zurück in µSv/h
        return np.sqrt(mean_squared_error(yva * y_std + y_mean, yva_pred))
    return objective

In [ ]:
# baseline MLP, 6 rotations

mlp_results, mlp_best_params_by_split, mlp_predictions_all = [], [], []

for train_p, val_p, test_p in tqdm(splits, desc="MLP rotations"):
    split_name = f"{train_p[-1]}{val_p[-1]}{test_p[-1]}"
    tr = df[df["partition"] == train_p]; va = df[df["partition"] == val_p]; te = df[df["partition"] == test_p]

    x_scaler = StandardScaler().fit(tr[feature_cols].values)
    Xtr_s = x_scaler.transform(tr[feature_cols].values)
    Xva_s = x_scaler.transform(va[feature_cols].values)
    Xte_s = x_scaler.transform(te[feature_cols].values)

    y_mean, y_std = tr[target_col].mean(), tr[target_col].std()
    ytr = (tr[target_col].values - y_mean) / y_std
    yva = (va[target_col].values - y_mean) / y_std
    y_test = te[target_col].values

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_objective(Xtr_s, ytr, Xva_s, yva, y_mean, y_std),   # neue Signatur
                   n_trials=N_TRIALS, show_progress_bar=False)
    bp = study.best_params

    seed_everything(42)
    model = make_mlp(Xtr_s.shape[1], bp["n_layers"], bp["d_hidden"], bp["dropout"])
    model, _ = train_nn(model, Xtr_s, ytr, Xva_s, yva,
                        lr=bp["lr"], weight_decay=bp["weight_decay"],
                        batch_size=bp["batch_size"], max_epochs=MAX_EPOCHS, patience=PATIENCE)
    y_test_pred = predict_nn(model, Xte_s) * y_std + y_mean

    test_m = evaluate_regression(y_test, y_test_pred)
    mlp_results.append({"Model": "MLP", "Split": split_name,
                        "Train_partition": train_p, "Validation_partition": val_p,
                        "Test_partition": test_p, **{f"Test_{k}": v for k, v in test_m.items()}})
    mlp_best_params_by_split.append({"Split": split_name, **bp,
                                     "Best_Validation_RMSE": study.best_value})

    pred_df = pd.DataFrame({"Model": "MLP", "Split": split_name,
                            "Train_partition": train_p, "Validation_partition": val_p,
                            "Test_partition": test_p, "y_true": y_test, "y_pred": y_test_pred})
    pred_df["residual"]   = pred_df["y_pred"] - pred_df["y_true"]
    pred_df["abs_error"]  = pred_df["residual"].abs()
    pred_df["sq_error"]   = pred_df["residual"] ** 2
    pred_df["underpred"]  = pred_df["y_pred"] < pred_df["y_true"]
    pred_df["tail_ge_15"] = pred_df["y_true"] >= 15
    pred_df["tail_ge_20"] = pred_df["y_true"] >= 20
    pred_df["pred_ge_15"] = pred_df["y_pred"] >= 15
    pred_df["pred_ge_20"] = pred_df["y_pred"] >= 20
    mlp_predictions_all.append(pred_df)

    pd.concat(mlp_predictions_all, ignore_index=True).to_csv("mlp_predictions_progress.csv", index=False)

print("Fertig.")

MLP rotations:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
# save results
mlp_results = pd.DataFrame(mlp_results)
mlp_best_params_by_split = pd.DataFrame(mlp_best_params_by_split)
mlp_predictions_all = pd.concat(mlp_predictions_all, ignore_index=True)
mlp_results.to_csv("mlp_results_final.csv", index=False)
mlp_predictions_all.to_csv("mlp_predictions_all.csv", index=False)
print("Rows:", len(mlp_predictions_all), "(erwartet", 2*len(df), ")")
mlp_results

In [ ]:
def regime_summary(pred_df, label, thr=20.0):
    out = []
    for split, g in pred_df.groupby("Split"):
        for regime, sub in {"Overall": g, "Tail(>=20)": g[g["y_true"] >= thr]}.items():
            m = regression_metrics_subset(sub)
            out.append({"Model": label, "Split": split, "Regime": regime, **m})
    return pd.DataFrame(out)

all_models = pd.concat([
    regime_summary(xgb_predictions_all,  "XGBoost"),
    regime_summary(xgbw_predictions_all, "XGBoost-Weighted"),
    regime_summary(mlp_predictions_all,  "MLP"),
], ignore_index=True)
all_models["Model"] = pd.Categorical(
    all_models["Model"], ["XGBoost", "XGBoost-Weighted", "MLP"], ordered=True)

compare_all = (all_models.groupby(["Regime", "Model"], observed=True)
               [["RMSE","MAE","Bias","Underestimation_%"]].agg(["mean","std"]).round(3))
compare_all

In [ ]:
# Error patterns: Predicted vs. Actual — Baseline MLP 
import matplotlib.pyplot as plt

TAIL_THR = 20.0
p = mlp_predictions_all
is_tail = p["y_true"] >= TAIL_THR
resid = p["y_pred"] - p["y_true"]
hi = max(p["y_true"].max(), p["y_pred"].max()) * 1.02

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.scatter(p.loc[~is_tail,"y_true"], p.loc[~is_tail,"y_pred"], s=6, alpha=0.12, label="Bulk (<20)")
ax.scatter(p.loc[is_tail,"y_true"],  p.loc[is_tail,"y_pred"],  s=12, alpha=0.5,
           color="crimson", label="Tail (≥20)")
ax.plot([0, hi], [0, hi], "k--", lw=1, label="perfect (y=x)")
ax.axvline(TAIL_THR, color="grey", ls=":", lw=1); ax.axhline(TAIL_THR, color="grey", ls=":", lw=1)
ax.set(xlim=(0,hi), ylim=(0,hi), xlabel="Actual ARMAS (µSv/h)",
       ylabel="Predicted ARMAS (µSv/h)", title="MLP baseline — Predicted vs. Actual")
ax.set_aspect("equal", adjustable="box"); ax.legend(loc="upper left", fontsize=9)

ax = axes[1]
ax.scatter(p.loc[~is_tail,"y_true"], resid[~is_tail], s=6, alpha=0.12)
ax.scatter(p.loc[is_tail,"y_true"],  resid[is_tail],  s=12, alpha=0.5, color="crimson")
ax.axhline(0, color="k", lw=1); ax.axvline(TAIL_THR, color="grey", ls=":", lw=1)
ax.set(xlabel="Actual ARMAS (µSv/h)", ylabel="Residual (pred − actual)",
       title="MLP baseline — Residuals vs. Actual")

plt.tight_layout()
plt.savefig("fig_mlp_baseline_errors.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Confusion Matrix + Recall — Baseline MLP 
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

TAIL_THR = 20.0
p = mlp_predictions_all
yt = (p["y_true"] >= TAIL_THR).astype(int)
yp = (p["y_pred"] >= TAIL_THR).astype(int)
cm = confusion_matrix(yt, yp, labels=[0, 1])

# Recall/Precision/F1 pro Split, mean ± std
rows = []
for split, g in p.groupby("Split"):
    a = (g["y_true"] >= TAIL_THR).astype(int); b = (g["y_pred"] >= TAIL_THR).astype(int)
    rows.append({"Split": split,
                 "Precision": precision_score(a, b, zero_division=0),
                 "Recall":    recall_score(a, b, zero_division=0),
                 "F1":        f1_score(a, b, zero_division=0)})
clf_mlp = pd.DataFrame(rows)
print("MLP baseline — Mean ± Std über 6 Splits:")
print(clf_mlp[["Precision","Recall","F1"]].agg(["mean","std"]).round(3))

# Heatmap
cmn = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5, 4.2))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks([0,1], ["pred <20","pred ≥20"]); ax.set_yticks([0,1], ["actual <20","actual ≥20"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]:,}\n({cmn[i,j]:.1%})", ha="center", va="center",
                fontsize=11, fontweight="bold", color="white" if cmn[i,j] > 0.5 else "black")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="row-normalised rate")
ax.set_title("Tail detection (≥20 µSv/h) — MLP baseline", fontsize=11)
plt.tight_layout(); plt.savefig("fig_mlp_baseline_confusion.pdf", bbox_inches="tight"); plt.show()

## Weighted Mlp

In [ ]:
# Weighted-MLP

TAIL_THR = 20.0
Y_CLIP = float(df[target_col].max()) * 1.5   # NEU: Sicherheitsnetz gegen expm1-Explosion (≈ 63.6)

def tail_rmse(y_true, y_pred, thr=TAIL_THR):
    m = np.asarray(y_true) >= thr
    if m.sum() == 0:
        return np.inf
    e = np.asarray(y_pred)[m] - np.asarray(y_true)[m]
    return np.sqrt(np.mean(e ** 2))

def make_objective_weighted(Xtr_s, ytr_log, Xva_s, y_val_orig, w_tr):
    """tail-aware: minimiere Validation-TAIL-RMSE in µSv/h (log1p-Target + Gewichte)."""
    def objective(trial):
        n_layers   = trial.suggest_int("n_layers", 1, 5)
        d_hidden   = trial.suggest_categorical("d_hidden", [64, 128, 256, 512])
        dropout    = trial.suggest_float("dropout", 0.0, 0.5)
        lr         = trial.suggest_float("lr", 1e-4, 3e-3, log=True)   # GEÄNDERT: 1e-2 -> 3e-3
        wd         = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
        batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])
        seed_everything(42)
        model = make_mlp(Xtr_s.shape[1], n_layers, d_hidden, dropout)
        model, _ = train_nn(model, Xtr_s, ytr_log, Xva_s, np.log1p(y_val_orig), w_tr=w_tr,
                            lr=lr, weight_decay=wd, batch_size=batch_size,
                            max_epochs=MAX_EPOCHS, patience=PATIENCE)
        yva_pred = np.clip(np.expm1(predict_nn(model, Xva_s)), 0, Y_CLIP)   # GEÄNDERT: + clip
        return tail_rmse(y_val_orig, yva_pred)
    return objective

In [ ]:
# After fix 1 Modell, lr=3e-3 + Clip
train_p, val_p, test_p = splits[0]
tr = df[df["partition"]==train_p]; va = df[df["partition"]==val_p]
xs = StandardScaler().fit(tr[feature_cols].values)
Xtr_s, Xva_s = xs.transform(tr[feature_cols].values), xs.transform(va[feature_cols].values)
y_tr, y_va = tr[target_col].values, va[target_col].values

seed_everything(42)
model = make_mlp(Xtr_s.shape[1], 2, 128, 0.1)
model, _ = train_nn(model, Xtr_s, np.log1p(y_tr), Xva_s, np.log1p(y_va),
                    w_tr=compute_density_weights(y_tr),
                    lr=3e-3, weight_decay=1e-5, batch_size=256, max_epochs=200, patience=16)
raw_pred = np.expm1(predict_nn(model, Xva_s))
y_va_pred = np.clip(raw_pred, 0, Y_CLIP)
print(f"y_pred max: {y_va_pred.max():.2f}   (Y_CLIP = {Y_CLIP:.1f}, y_true max = {y_va.max():.2f})")
print(f"Vorhersagen am Clip (>{Y_CLIP:.0f}): {(raw_pred > Y_CLIP).sum()}")

In [ ]:
# Weighted-MLP:full (corrected with clip)
mlpw_results, mlpw_best_params_by_split, mlpw_predictions_all = [], [], []

for train_p, val_p, test_p in tqdm(splits, desc="Weighted-MLP rotations"):
    split_name = f"{train_p[-1]}{val_p[-1]}{test_p[-1]}"
    tr = df[df["partition"] == train_p]; va = df[df["partition"] == val_p]; te = df[df["partition"] == test_p]

    x_scaler = StandardScaler().fit(tr[feature_cols].values)
    Xtr_s = x_scaler.transform(tr[feature_cols].values)
    Xva_s = x_scaler.transform(va[feature_cols].values)
    Xte_s = x_scaler.transform(te[feature_cols].values)

    y_train, y_val, y_test = tr[target_col].values, va[target_col].values, te[target_col].values
    ytr_log = np.log1p(y_train)
    w_tr    = compute_density_weights(y_train)

    study = optuna.create_study(direction="minimize",
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(make_objective_weighted(Xtr_s, ytr_log, Xva_s, y_val, w_tr),
                   n_trials=N_TRIALS, show_progress_bar=False)
    bp = study.best_params

    seed_everything(42)
    model = make_mlp(Xtr_s.shape[1], bp["n_layers"], bp["d_hidden"], bp["dropout"])
    model, _ = train_nn(model, Xtr_s, ytr_log, Xva_s, np.log1p(y_val), w_tr=w_tr,
                        lr=bp["lr"], weight_decay=bp["weight_decay"],
                        batch_size=bp["batch_size"], max_epochs=MAX_EPOCHS, patience=PATIENCE)
    y_test_pred = np.clip(np.expm1(predict_nn(model, Xte_s)), 0, Y_CLIP)   # Clip auch auf Test

    test_m = evaluate_regression(y_test, y_test_pred)
    mlpw_results.append({"Model": "MLP-Weighted", "Split": split_name,
                         "Train_partition": train_p, "Validation_partition": val_p,
                         "Test_partition": test_p, **{f"Test_{k}": v for k, v in test_m.items()}})
    mlpw_best_params_by_split.append({"Split": split_name, **bp,
                                      "Best_Validation_TailRMSE": study.best_value})

    pred_df = pd.DataFrame({"Model": "MLP-Weighted", "Split": split_name,
                            "Train_partition": train_p, "Validation_partition": val_p,
                            "Test_partition": test_p, "y_true": y_test, "y_pred": y_test_pred})
    pred_df["residual"]   = pred_df["y_pred"] - pred_df["y_true"]
    pred_df["abs_error"]  = pred_df["residual"].abs()
    pred_df["sq_error"]   = pred_df["residual"] ** 2
    pred_df["underpred"]  = pred_df["y_pred"] < pred_df["y_true"]
    pred_df["tail_ge_15"] = pred_df["y_true"] >= 15
    pred_df["tail_ge_20"] = pred_df["y_true"] >= 20
    pred_df["pred_ge_15"] = pred_df["y_pred"] >= 15
    pred_df["pred_ge_20"] = pred_df["y_pred"] >= 20
    mlpw_predictions_all.append(pred_df)

    pd.concat(mlpw_predictions_all, ignore_index=True).to_csv("mlpw_predictions_progress.csv", index=False)

print("Done")

In [ ]:
# save results

mlpw_results = pd.DataFrame(mlpw_results)
mlpw_best_params_by_split = pd.DataFrame(mlpw_best_params_by_split)
mlpw_predictions_all = pd.concat(mlpw_predictions_all, ignore_index=True)
mlpw_results.to_csv("mlpw_results_final.csv", index=False)
mlpw_predictions_all.to_csv("mlpw_predictions_all.csv", index=False)

print("Rows:", len(mlpw_predictions_all), "(erwartet", 2*len(df), ")")
print("y_pred max:", round(mlpw_predictions_all["y_pred"].max(), 2), "(sollte ~42 sein, nicht 1500)")
mlpw_results

In [ ]:
(mlpw_predictions_all["y_pred"] >= 63).sum() 

In [ ]:
# 4 model compairson (overall and tail)
def regime_summary(pred_df, label, thr=20.0):
    out = []
    for split, g in pred_df.groupby("Split"):
        for regime, sub in {"Overall": g, "Tail(>=20)": g[g["y_true"] >= thr]}.items():
            m = regression_metrics_subset(sub)
            out.append({"Model": label, "Split": split, "Regime": regime, **m})
    return pd.DataFrame(out)

all_models = pd.concat([
    regime_summary(xgb_predictions_all,  "XGBoost"),
    regime_summary(xgbw_predictions_all, "XGBoost-Weighted"),
    regime_summary(mlp_predictions_all,  "MLP"),
    regime_summary(mlpw_predictions_all, "MLP-Weighted"),
], ignore_index=True)
all_models["Model"] = pd.Categorical(
    all_models["Model"], ["XGBoost","XGBoost-Weighted","MLP","MLP-Weighted"], ordered=True)
compare_all = (all_models.groupby(["Regime","Model"], observed=True)
               [["RMSE","MAE","Bias","Underestimation_%"]].agg(["mean","std"]).round(3))
compare_all

In [ ]:
# Recall MLP-Weighted
from sklearn.metrics import recall_score, precision_score, f1_score
rows = []
for split, g in mlpw_predictions_all.groupby("Split"):
    a = (g["y_true"] >= 20).astype(int); b = (g["y_pred"] >= 20).astype(int)
    rows.append({"Recall": recall_score(a,b,zero_division=0),
                 "Precision": precision_score(a,b,zero_division=0),
                 "F1": f1_score(a,b,zero_division=0)})
print("MLP-Weighted — Mean ± Std:")
print(pd.DataFrame(rows)[["Recall","Precision","F1"]].agg(["mean","std"]).round(3))

In [ ]:
# Predicted vs. Actual: Baseline-MLP vs. Weighted-MLP 
import matplotlib.pyplot as plt
TAIL_THR = 20.0
panels = [("MLP baseline", mlp_predictions_all), ("MLP weighted", mlpw_predictions_all)]
hi = max(mlp_predictions_all[["y_true","y_pred"]].max().max(),
         mlpw_predictions_all[["y_true","y_pred"]].max().max()) * 1.02

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
for ax, (title, p) in zip(axes, panels):
    is_tail = p["y_true"] >= TAIL_THR
    ax.scatter(p.loc[~is_tail,"y_true"], p.loc[~is_tail,"y_pred"], s=6, alpha=0.12, label="Bulk (<20)")
    ax.scatter(p.loc[is_tail,"y_true"],  p.loc[is_tail,"y_pred"],  s=12, alpha=0.5,
               color="crimson", label="Tail (≥20)")
    ax.plot([0, hi], [0, hi], "k--", lw=1, label="perfect (y=x)")
    ax.axvline(TAIL_THR, color="grey", ls=":", lw=1); ax.axhline(TAIL_THR, color="grey", ls=":", lw=1)
    ax.set(xlim=(0,hi), ylim=(0,hi), xlabel="Actual ARMAS (µSv/h)", title=title)
    ax.set_aspect("equal", adjustable="box")
axes[0].set_ylabel("Predicted ARMAS (µSv/h)")
axes[0].legend(loc="upper left", fontsize=9)
fig.suptitle("Predicted vs. Actual — MLP: effect of imbalance-aware weighting", y=1.02, fontsize=13)
plt.tight_layout(); plt.savefig("fig_mlp_pred_vs_actual.pdf", bbox_inches="tight"); plt.show()

In [ ]:
# feature importance

import matplotlib.pyplot as plt

def permutation_importance(predict_fn, X, y_true, feature_names, n_repeats=5, seed=42):
    rng = np.random.default_rng(seed)
    base = np.sqrt(mean_squared_error(y_true, predict_fn(X)))
    imp = []
    for j in range(X.shape[1]):
        scores = []
        for _ in range(n_repeats):
            Xp = X.copy()
            col = Xp[:, j].copy(); rng.shuffle(col); Xp[:, j] = col
            scores.append(np.sqrt(mean_squared_error(y_true, predict_fn(Xp))) - base)
        imp.append(np.mean(scores))
    return pd.DataFrame({"feature": feature_names, "importance": imp}).sort_values("importance", ascending=False)


shap_split = "231"
bp = mlp_best_params_by_split.set_index("Split").loc[shap_split]  # oder aus CSV laden
train_p, val_p, test_p = "P"+shap_split[0], "P"+shap_split[1], "P"+shap_split[2]
tr = df[df["partition"]==train_p]; va = df[df["partition"]==val_p]; te = df[df["partition"]==test_p]

xs = StandardScaler().fit(tr[feature_cols].values)
Xtr_s, Xva_s, Xte_s = (xs.transform(tr[feature_cols].values),
                       xs.transform(va[feature_cols].values),
                       xs.transform(te[feature_cols].values))
y_mean, y_std = tr[target_col].mean(), tr[target_col].std()
ytr = (tr[target_col].values-y_mean)/y_std
yva = (va[target_col].values-y_mean)/y_std
y_test = te[target_col].values

seed_everything(42)
model = make_mlp(Xtr_s.shape[1], int(bp["n_layers"]), int(bp["d_hidden"]), float(bp["dropout"]))
model, _ = train_nn(model, Xtr_s, ytr, Xva_s, yva,
                    lr=float(bp["lr"]), weight_decay=float(bp["weight_decay"]),
                    batch_size=int(bp["batch_size"]), max_epochs=MAX_EPOCHS, patience=PATIENCE)

predict_fn = lambda X: predict_nn(model, X) * y_std + y_mean
pi_mlp = permutation_importance(predict_fn, Xte_s, y_test, feature_cols, n_repeats=5)
pi_mlp.to_csv("mlp_permutation_importance.csv", index=False)

import matplotlib.pyplot as plt
top = pi_mlp.head(15).iloc[::-1]
plt.figure(figsize=(7,5))
plt.barh(top["feature"], top["importance"], color="#378ADD")
plt.xlabel("Increase in RMSE when feature is permuted (µSv/h)")
plt.title("Permutation importance — MLP baseline (split 231)")
plt.tight_layout(); plt.savefig("fig_mlp_permutation_importance.pdf", bbox_inches="tight"); plt.show()
pi_mlp.head(15)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
pi_mlp = pd.read_csv("mlp_permutation_importance.csv")
top = pi_mlp.head(15).iloc[::-1]
plt.figure(figsize=(7,5))
plt.barh(top["feature"], top["importance"], color="#1B6B47")   # dunkelgrün = Baseline
plt.xlabel("Increase in RMSE when feature is permuted (µSv/h)")
plt.title("Permutation importance — MLP baseline (split 231)")
plt.tight_layout(); plt.savefig("fig_mlp_permutation_importance.pdf", bbox_inches="tight"); plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

# Permutation-Importance-Funktion 
def permutation_importance(predict_fn, X, y_true, feature_names, n_repeats=5, seed=42):
    rng = np.random.default_rng(seed)
    base = np.sqrt(mean_squared_error(y_true, predict_fn(X)))
    imp = []
    for j in range(X.shape[1]):
        scores = []
        for _ in range(n_repeats):
            Xp = X.copy()
            col = Xp[:, j].copy(); rng.shuffle(col); Xp[:, j] = col
            scores.append(np.sqrt(mean_squared_error(y_true, predict_fn(Xp))) - base)
        imp.append(np.mean(scores))
    return pd.DataFrame({"feature": feature_names, "importance": imp}).sort_values("importance", ascending=False)

# Weighted-MLP training
shap_split = "231"
bp = mlpw_best_params_by_split.set_index("Split").loc[(shap_split)]
train_p, val_p, test_p = "P"+shap_split[0], "P"+shap_split[1], "P"+shap_split[2]
tr = df[df["partition"]==train_p]; va = df[df["partition"]==val_p]; te = df[df["partition"]==test_p]

xs = StandardScaler().fit(tr[feature_cols].values)
Xtr_s = xs.transform(tr[feature_cols].values)
Xva_s = xs.transform(va[feature_cols].values)
Xte_s = xs.transform(te[feature_cols].values)

y_train, y_val, y_test = tr[target_col].values, va[target_col].values, te[target_col].values
ytr_log = np.log1p(y_train)
w_tr    = compute_density_weights(y_train)

seed_everything(42)
model = make_mlp(Xtr_s.shape[1], int(bp["n_layers"]), int(bp["d_hidden"]), float(bp["dropout"]))
model, _ = train_nn(model, Xtr_s, ytr_log, Xva_s, np.log1p(y_val), w_tr=w_tr,
                    lr=float(bp["lr"]), weight_decay=float(bp["weight_decay"]),
                    batch_size=int(bp["batch_size"]), max_epochs=MAX_EPOCHS, patience=PATIENCE)

# Vorhersage = expm1(...) mit Clip, wie im Weighted-Training
Y_CLIP = float(df[target_col].max()) * 1.5
predict_fn = lambda X: np.clip(np.expm1(predict_nn(model, X)), 0, Y_CLIP)

pi_mlpw = permutation_importance(predict_fn, Xte_s, y_test, feature_cols, n_repeats=5)
pi_mlpw.to_csv("mlpw_permutation_importance.csv", index=False)

top = pi_mlpw.head(15).iloc[::-1]
plt.figure(figsize=(7,5))
plt.barh(top["feature"], top["importance"], color="#2E7D5B")
plt.xlabel("Increase in RMSE when feature is permuted (µSv/h)")
plt.title("Permutation importance — MLP weighted (split 231)")
plt.tight_layout(); plt.savefig("fig_mlpw_permutation_importance.pdf", bbox_inches="tight"); plt.show()
pi_mlpw.head(15)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
pi_mlpw = pd.read_csv("mlpw_permutation_importance.csv")
top = pi_mlpw.head(15).iloc[::-1]
plt.figure(figsize=(7,5))
plt.barh(top["feature"], top["importance"], color="#5FBF96")   # light green = weighted
plt.xlabel("Increase in RMSE when feature is permuted (µSv/h)")
plt.title("Permutation importance — MLP weighted (split 231)")
plt.tight_layout(); plt.savefig("fig_mlpw_permutation_importance.pdf", bbox_inches="tight"); plt.show()